In [3]:
%reload_ext autoreload
%autoreload 2

%matplotlib inline

import os
import sys
from pathlib import Path

import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Ensure we run from project root so `data/raw` resolves correctly
if Path.cwd().name == "notebooks":
    os.chdir("..")


project_root = str(Path.cwd())
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"CWD: {Path.cwd()}")

CWD: c:\Users\reymo\.cursor\Almgren_Chriss_problem


In [5]:
sys.path.append(os.path.abspath('./src'))

from data_downloader import MicrostructureDownloader
from dataset_builder import build_ml_dataset
from features import Market_State_Engineer, Execution_Parameter_Regressor

In [7]:

# 1. Ingest and Prepare Data
downloader = MicrostructureDownloader(data_dir="data/raw")
df = downloader.fetch_intraday(ticker="AAPL", start_date="2026-09-13", end_date="2026-09-27", interval="5m")

engineer = Market_State_Engineer(window=6)
X, y_sigma, y_eta = build_ml_dataset(df, engineer, m_forward_periods=3)

# 2. Chronological Train-Test Split (80/20)
train_ratio = 0.8
split_idx = int(len(X) * train_ratio)

X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_sigma_train, y_sigma_test = y_sigma.iloc[:split_idx], y_sigma.iloc[split_idx:]
y_eta_train, y_eta_test = y_eta.iloc[:split_idx], y_eta.iloc[split_idx:]

# 3. Instantiate and Train Custom Models
alphas = np.logspace(-4, 2, 20)
sigma_model = Execution_Parameter_Regressor(alphas=alphas)
eta_model = Execution_Parameter_Regressor(alphas=alphas)

sigma_model.fit(X_train, y_sigma_train)
eta_model.fit(X_train, y_eta_train)

# 4. Predict and Transform Back to Real Space
sigma_preds_log = np.array(sigma_model.predict(X_test)).flatten()
eta_preds_log = np.array(eta_model.predict(X_test)).flatten()

sigma_preds_real = np.exp(sigma_preds_log)
eta_preds_real = np.exp(eta_preds_log)

y_sigma_test_real = np.exp(y_sigma_test)
y_eta_test_real = np.exp(y_eta_test)

# 5. Construct Naive Baselines (10-period SMA in log-space, exponentiated)
sma_window = 10
naive_sigma_log = y_sigma.shift(1).rolling(window=sma_window).mean().iloc[split_idx:]
naive_eta_log = y_eta.shift(1).rolling(window=sma_window).mean().iloc[split_idx:]

# Align indices to remove NaNs from the SMA window boundary
valid_idx = naive_sigma_log.dropna().index

y_sigma_aligned = y_sigma_test_real.loc[valid_idx]
y_eta_aligned = y_eta_test_real.loc[valid_idx]

sigma_preds_aligned = pd.Series(sigma_preds_real, index=X_test.index).loc[valid_idx]
eta_preds_aligned = pd.Series(eta_preds_real, index=X_test.index).loc[valid_idx]

naive_sigma_aligned = np.exp(naive_sigma_log.loc[valid_idx])
naive_eta_aligned = np.exp(naive_eta_log.loc[valid_idx])

# 6. Compute Metrics
sigma_mse_custom = mean_squared_error(y_sigma_aligned, sigma_preds_aligned)
sigma_mse_naive = mean_squared_error(y_sigma_aligned, naive_sigma_aligned)
sigma_mae_custom = mean_absolute_error(y_sigma_aligned, sigma_preds_aligned)
sigma_mae_naive = mean_absolute_error(y_sigma_aligned, naive_sigma_aligned)

eta_mse_custom = mean_squared_error(y_eta_aligned, eta_preds_aligned)
eta_mse_naive = mean_squared_error(y_eta_aligned, naive_eta_aligned)
eta_mae_custom = mean_absolute_error(y_eta_aligned, eta_preds_aligned)
eta_mae_naive = mean_absolute_error(y_eta_aligned, naive_eta_aligned)

# 7. Print Results
print("--- Out-of-Sample Performance: Volatility (\u03c3) ---")
print(f"Custom Model MSE: {sigma_mse_custom:.4e} | Naive MSE: {sigma_mse_naive:.4e}")
print(f"Custom Model MAE: {sigma_mae_custom:.4e} | Naive MAE: {sigma_mae_naive:.4e}")
print(f"MSE Improvement: {((sigma_mse_naive - sigma_mse_custom) / sigma_mse_naive) * 100:.2f}%\n")

print("--- Out-of-Sample Performance: Liquidity (\u03b7) ---")
print(f"Custom Model MSE: {eta_mse_custom:.4e} | Naive MSE: {eta_mse_naive:.4e}")
print(f"Custom Model MAE: {eta_mae_custom:.4e} | Naive MAE: {eta_mae_naive:.4e}")
print(f"MSE Improvement: {((eta_mse_naive - eta_mse_custom) / eta_mse_naive) * 100:.2f}%")

--- Out-of-Sample Performance: Volatility (σ) ---
Custom Model MSE: 1.2786e-01 | Naive MSE: 9.9109e-02
Custom Model MAE: 2.6829e-01 | Naive MAE: 2.3666e-01
MSE Improvement: -29.01%

--- Out-of-Sample Performance: Liquidity (η) ---
Custom Model MSE: 2.4104e-18 | Naive MSE: 2.5603e-18
Custom Model MAE: 1.1065e-09 | Naive MAE: 1.1499e-09
MSE Improvement: 5.85%
